# N31 — Strategy Orchestrator

End-to-end multi-agent supervisor. Integrates N25–N30 sub-agents through three
processing layers: dynamic MoE-style routing, Monte Carlo simulation over the
probabilistic outputs, and structured LLM synthesis of the final recommendation.

**Architecture**
- Layer 1: dynamic routing — decides which sub-agents to call each lap
- Layer 2: Monte Carlo simulation — samples from each sub-agent's uncertainty outputs
- Layer 3: LLM synthesis — aggregates all reasoning + MC scores into `StrategyRecommendation`

**References**: Heilmeier et al. (2020) ApplSci 10/4229 (MC motorsport sim),
Wang et al. (2024) MoA arXiv:2406.04692 (reasoning aggregation pattern),
Liu et al. (2024) DeLLMa arXiv:2402.02392 (decision under uncertainty with LLM).


---

## Step 0 — Setup & model loading

Imports, repo root, LLM client, FastF1 cache. `OrchestratorCFG` holds the
orchestrator's runtime constants: MC simulation parameters, MoE routing
thresholds, and the LLM client used in Layer 3 synthesis.

| Constant | Value | Purpose |
|---|---|---|
| `n_sim` | 500 | MC draws per strategy candidate |
| `sc_prob_threshold` | 0.30 | activates N30 if N27.sc_prob_3lap exceeds this |
| `risk_tolerance_default` | 0.5 | α in `score = α·E[S] + (1−α)·P10[S]` |


In [1]:
import json, sys
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import fastf1

repo_root = Path.cwd()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI


c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
@dataclass
class OrchestratorCFG:
    """Runtime configuration for the Strategy Orchestrator (N31).

    n_sim controls Monte Carlo draws per strategy candidate in Layer 2. 500 draws
    keep variance of the mean below 0.01 position units within lap-level latency.

    sc_prob_threshold is the N27.sc_prob_3lap cutoff above which N30 is activated
    to retrieve safety-car regulation context for the pit decision.

    risk_tolerance_default (α) weights expected value vs worst-case in the MC
    score: score(S) = α·E[S] + (1−α)·P10[S]. α=1.0 aggressive, α=0.0 conservative.

    temperature=0.0 ensures deterministic structured output from Layer 3 LLM.
    """

    model_name: str = "local-model"
    base_url: str = "http://localhost:1234/v1"
    temperature: float = 0.0
    n_sim: int = 500
    sc_prob_threshold: float = 0.30
    risk_tolerance_default: float = 0.5

    def __post_init__(self):
        root = Path.cwd()
        while not (root / ".git").exists():
            root = root.parent

        fastf1.Cache.enable_cache(str(root / "data" / "cache" / "fastf1"))

        self.export_dir = root / "data" / "models" / "agents"
        self.export_dir.mkdir(parents=True, exist_ok=True)

        self.llm = ChatOpenAI(
            model=self.model_name,
            base_url=self.base_url,
            api_key="lm-studio",
            temperature=self.temperature,
        )

In [3]:
CFG = OrchestratorCFG()

print(f"LLM          : {CFG.model_name} @ {CFG.base_url}")
print(f"N_SIM        : {CFG.n_sim}")
print(f"SC threshold : {CFG.sc_prob_threshold}")
print(f"α default    : {CFG.risk_tolerance_default}")
print(f"EXPORT_DIR   : {CFG.export_dir}")

LLM          : local-model @ http://localhost:1234/v1
N_SIM        : 500
SC threshold : 0.3
α default    : 0.5
EXPORT_DIR   : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\models\agents
